In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install scikit-posthocs

In [17]:
import pandas as pd
import numpy as np
from scipy import stats
from scikit_posthocs import posthoc_dunn

In [18]:
base = "/content/drive/MyDrive/06-Green Washing AI/analysis/generated_content/"
# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv(base + "evaluated_greenwashing_results.csv")  # adjust sep if needed

CONDITIONS = ["no_context", "context_only", "kg_context"]
DIMS       = ["score_vagueness", "score_misleading", "score_concealment",
              "score_overselling", "score_irrelevance"]
DIM_LABELS = ["Vagueness", "Misleading", "Concealment", "Overselling", "Irrelevance"]
TOTAL      = "total_greenwashing_score"

In [19]:
df.head()

,run_id,Documentation_richness,company_name,prompt_type,grounding_condition,generated_text,total_greenwashing_score,score_vagueness,score_misleading,score_concealment,...,repetition,model,input_tokens,output_tokens,timestamp,justification_vagueness,justification_misleading,justification_concealment,justification_overselling,justification_irrelevance
0,Faiksonmez En__vagueness__no_context__rep1,Low,Faik Sönmez,vagueness,no_context,"At Faik Sönmez, every collection begins with a...",7,3.0,2.0,0.0,...,1.0,gpt-5.1,[163],[234],2026-03-06T09:01:51.354456,"The text relies entirely on broad, undefined s...",The marketing assertions of an embedded enviro...,The text does not highlight a single narrow ec...,The language is highly emotive and grandiose (...,The text does not tout compliance with legal m...
1,Faiksonmez En__vagueness__context_only__rep1,Low,Faik Sönmez,vagueness,context_only,"For us at Faik Sönmez, caring for the planet b...",6,2.0,2.0,1.0,...,1.0,gpt-5.1,[626],[418],2026-03-06T09:02:08.525713,"The text uses broad, emotive sustainability la...",The generated copy introduces a temporal metri...,"The text emphasizes longevity, craftsmanship a...","The copy uses elevated, nature‑evoking languag...",The text does not tout legally required compli...
2,Faiksonmez En__vagueness__kg_context__rep1,Low,Faik Sönmez,vagueness,kg_context,"At Faik Sönmez, our vision of a more mindful f...",3,1.0,0.0,1.0,...,1.0,gpt-5.1,[1446],[1236],2026-03-06T09:03:34.264570,"The text uses broad, feel-good sustainability ...","The numeric metrics in the text (35,000 fabric...","The copy highlights fit, quality fabrics, and ...","The text employs elevated, emotive language to...","The statements concern design, fit, fabric sel..."
3,Faiksonmez En__misleading__no_context__rep1,Low,Faik Sönmez,misleading,no_context,"At Faik Sönmez, we are turning ambition into m...",8,2.0,3.0,1.0,...,1.0,gpt-5.1,[203],[944],2026-03-06T09:04:40.959127,The copy mixes concrete-sounding claims with a...,High presence of unverifiable metrics and cert...,The text emphasizes certified items and select...,"The language inflates impact (e.g., 'turning a...",The text does not boast about basic legal comp...
4,Faiksonmez En__misleading__context_only__rep1,Low,Faik Sönmez,misleading,context_only,"At Faik Sönmez, we believe true sustainability...",7,2.0,2.0,2.0,...,1.0,gpt-5.1,[666],[650],2026-03-06T09:04:52.397324,The text uses broad sustainability language (e...,The generated text includes a specific tempora...,The copy foregrounds positive attributes (fabr...,The language frames the brand's design and sel...,The text does not present claims that merely r...


In [20]:
# ── Descriptive statistics ────────────────────────────────────────────────────
print("=" * 70)
print("DESCRIPTIVE STATISTICS — Median [IQR] per condition")
print("=" * 70)

all_cols = [TOTAL] + DIMS
all_labels = ["Total"] + DIM_LABELS

for col, label in zip(all_cols, all_labels):
    print(f"\n  {label}")
    print(f"  {'Condition':<15} {'Median':>8} {'IQR (Q1-Q3)':>20} {'Mean':>8}")
    print(f"  {'-'*55}")
    for cond in CONDITIONS:
        vals = df[df["grounding_condition"] == cond][col]
        q1, q3 = vals.quantile([0.25, 0.75])
        print(f"  {cond:<15} {vals.median():>8.2f} "
              f"  [{q1:.2f}, {q3:.2f}]       {vals.mean():>8.2f}")

DESCRIPTIVE STATISTICS — Median [IQR] per condition

  Total
  Condition         Median          IQR (Q1-Q3)     Mean
  -------------------------------------------------------
  no_context          8.00   [7.00, 9.00]           8.00
  context_only        5.50   [3.00, 7.25]           5.25
  kg_context          3.00   [2.75, 5.00]           3.60

  Vagueness
  Condition         Median          IQR (Q1-Q3)     Mean
  -------------------------------------------------------
  no_context          2.00   [1.00, 3.00]           1.90
  context_only        1.00   [1.00, 2.00]           1.30
  kg_context          1.00   [1.00, 2.00]           1.30

  Misleading
  Condition         Median          IQR (Q1-Q3)     Mean
  -------------------------------------------------------
  no_context          2.00   [2.00, 3.00]           2.30
  context_only        2.00   [0.00, 2.00]           1.25
  kg_context          0.00   [0.00, 0.00]           0.20

  Concealment
  Condition         Median          IQR

In [21]:
# ── Kruskal-Wallis ────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("KRUSKAL-WALLIS TEST — H statistic and p-value")
print("=" * 70)
print(f"  {'Dimension':<15} {'H':>8} {'p-value':>12} {'Sig':>6}")
print(f"  {'-'*45}")

kw_results = {}
for col, label in zip(all_cols, all_labels):
    groups = [df[df["grounding_condition"] == c][col].values for c in CONDITIONS]
    H, p = stats.kruskal(*groups)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    kw_results[label] = {"H": H, "p": p, "sig": sig}
    print(f"  {label:<15} {H:>8.3f} {p:>12.4f} {sig:>6}")


KRUSKAL-WALLIS TEST — H statistic and p-value
  Dimension              H      p-value    Sig
  ---------------------------------------------
  Total             27.915       0.0000    ***
  Vagueness          6.639       0.0362      *
  Misleading        32.522       0.0000    ***
  Concealment        2.200       0.3329   n.s.
  Overselling       20.380       0.0000    ***
  Irrelevance        1.465       0.4806   n.s.


In [23]:
# ── Dunn post-hoc (Bonferroni) ────────────────────────────────────────────────
print("\n" + "=" * 70)
print("DUNN POST-HOC TEST (Bonferroni correction)")
print("Pairwise p-values: no_context vs context_only vs kg_context")
print("=" * 70)

df_clean = df[df["grounding_condition"].isin(CONDITIONS)].copy()

pairs = [("no_context", "context_only"),
         ("no_context", "kg_context"),
         ("context_only", "kg_context")]

for col, label in zip(all_cols, all_labels):
    print(f"\n  {label}")
    dunn = posthoc_dunn(df_clean, val_col=col, group_col="grounding_condition",
                        p_adjust="bonferroni")
    print(f"  {'Pair':<35} {'p-value':>10} {'Sig':>6}")
    print(f"  {'-'*55}")
    for c1, c2 in pairs:
        p = dunn.loc[c1, c2]
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
        print(f"  {c1+' vs '+c2:<35} {p:>10.4f} {sig:>6}")


DUNN POST-HOC TEST (Bonferroni correction)
Pairwise p-values: no_context vs context_only vs kg_context

  Total
  Pair                                   p-value    Sig
  -------------------------------------------------------
  no_context vs context_only              0.0027     **
  no_context vs kg_context                0.0000    ***
  context_only vs kg_context              0.1731   n.s.

  Vagueness
  Pair                                   p-value    Sig
  -------------------------------------------------------
  no_context vs context_only              0.0770   n.s.
  no_context vs kg_context                0.0770   n.s.
  context_only vs kg_context              1.0000   n.s.

  Misleading
  Pair                                   p-value    Sig
  -------------------------------------------------------
  no_context vs context_only              0.0125      *
  no_context vs kg_context                0.0000    ***
  context_only vs kg_context              0.0137      *

  Concealment

In [24]:

# ── Effect size (eta-squared) ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("EFFECT SIZE — Eta-squared (η²) for Kruskal-Wallis")
print("η² = (H - k + 1) / (n - k)  where k = number of groups")
print("=" * 70)
print(f"  {'Dimension':<15} {'η²':>8} {'Interpretation'}")
print(f"  {'-'*45}")

n = len(df)
k = len(CONDITIONS)
for label in all_labels:
    H = kw_results[label]["H"]
    eta2 = (H - k + 1) / (n - k)
    eta2 = max(0, eta2)  # clip to 0 if negative
    interp = "large" if eta2 >= 0.14 else "medium" if eta2 >= 0.06 else "small"
    print(f"  {label:<15} {eta2:>8.3f} {interp}")


EFFECT SIZE — Eta-squared (η²) for Kruskal-Wallis
η² = (H - k + 1) / (n - k)  where k = number of groups
  Dimension             η² Interpretation
  ---------------------------------------------
  Total              0.447 large
  Vagueness          0.080 medium
  Misleading         0.526 large
  Concealment        0.003 small
  Overselling        0.317 large
  Irrelevance        0.000 small


In [26]:
# ── Kruskal-Wallis by documentation richness ──────────────────────────────────
print("\n" + "=" * 70)
print("KRUSKAL-WALLIS TEST — By documentation richness (High vs Low SIRS)")
print("=" * 70)

RICHNESS = ["High", "Low"]
df_rich = df[df["Documentation_richness"].isin(RICHNESS)].copy()

print(f"\n  DESCRIPTIVE — Median [IQR] per richness group")
print(f"  {'Dimension':<15} {'Group':<8} {'Median':>8} {'IQR':>18} {'Mean':>8}")
print(f"  {'-'*60}")

kw_rich_results = {}
for col, label in zip(all_cols, all_labels):
    for r in RICHNESS:
        vals = df_rich[df_rich["Documentation_richness"] == r][col]
        q1, q3 = vals.quantile([0.25, 0.75])
        print(f"  {label:<15} {r:<8} {vals.median():>8.2f}  [{q1:.2f}, {q3:.2f}]  {vals.mean():>8.2f}")

print(f"\n  {'Dimension':<15} {'H':>8} {'p-value':>12} {'Sig':>6} {'η²':>8}")
print(f"  {'-'*55}")

for col, label in zip(all_cols, all_labels):
    groups = [df_rich[df_rich["Documentation_richness"] == r][col].values for r in RICHNESS]
    H, p = stats.kruskal(*groups)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
    n_r = len(df_rich)
    k_r = len(RICHNESS)
    eta2 = max(0, (H - k_r + 1) / (n_r - k_r))
    kw_rich_results[label] = {"H": H, "p": p, "sig": sig, "eta2": eta2}
    print(f"  {label:<15} {H:>8.3f} {p:>12.4f} {sig:>6} {eta2:>8.3f}")


KRUSKAL-WALLIS TEST — By documentation richness (High vs Low SIRS)

  DESCRIPTIVE — Median [IQR] per richness group
  Dimension       Group      Median                IQR     Mean
  ------------------------------------------------------------
  Total           High         5.00  [2.25, 7.00]      4.70
  Total           Low          7.00  [5.00, 8.00]      6.53
  Vagueness       High         1.00  [1.00, 1.00]      1.20
  Vagueness       Low          2.00  [1.00, 2.00]      1.80
  Misleading      High         0.00  [0.00, 2.00]      0.87
  Misleading      Low          2.00  [1.00, 2.00]      1.63
  Concealment     High         1.00  [0.00, 2.00]      1.03
  Concealment     Low          1.00  [0.25, 2.00]      1.20
  Overselling     High         1.00  [1.00, 2.00]      1.33
  Overselling     Low          2.00  [1.00, 2.00]      1.63
  Irrelevance     High         0.00  [0.00, 0.00]      0.27
  Irrelevance     Low          0.00  [0.00, 0.00]      0.27

  Dimension              H      p-v

In [27]:
# ── Interaction: richness × grounding condition ───────────────────────────────
print("\n" + "=" * 70)
print("INTERACTION ANALYSIS — Grounding condition effect within each richness group")
print("(Kruskal-Wallis run separately for High SIRS and Low SIRS companies)")
print("=" * 70)

for richness_level in RICHNESS:
    df_sub = df_rich[df_rich["Documentation_richness"] == richness_level].copy()
    print(f"\n  >>> {richness_level} SIRS companies (n={len(df_sub)})")
    print(f"  {'Dimension':<15} {'H':>8} {'p-value':>12} {'Sig':>6} {'η²':>8}")
    print(f"  {'-'*55}")

    for col, label in zip(all_cols, all_labels):
        groups = [df_sub[df_sub["grounding_condition"] == c][col].values for c in CONDITIONS]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2:
            print(f"  {label:<15} {'insufficient data':>40}")
            continue
        H, p = stats.kruskal(*groups)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
        n_s = len(df_sub)
        k_s = len(groups)
        eta2 = max(0, (H - k_s + 1) / (n_s - k_s))
        print(f"  {label:<15} {H:>8.3f} {p:>12.4f} {sig:>6} {eta2:>8.3f}")

    print(f"\n  Dunn post-hoc within {richness_level} SIRS:")
    for col, label in zip(all_cols, all_labels):
        dunn = posthoc_dunn(df_sub, val_col=col, group_col="grounding_condition",
                            p_adjust="bonferroni")
        print(f"\n    {label}")
        for c1, c2 in pairs:
            try:
                p = dunn.loc[c1, c2]
                sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "n.s."
                print(f"    {c1+' vs '+c2:<35} {p:>10.4f} {sig:>6}")
            except KeyError:
                print(f"    {c1+' vs '+c2:<35} {'N/A':>10}")


INTERACTION ANALYSIS — Grounding condition effect within each richness group
(Kruskal-Wallis run separately for High SIRS and Low SIRS companies)

  >>> High SIRS companies (n=30)
  Dimension              H      p-value    Sig       η²
  -------------------------------------------------------
  Total             19.931       0.0000    ***    0.664
  Vagueness         15.086       0.0005    ***    0.485
  Misleading        22.703       0.0000    ***    0.767
  Concealment        1.765       0.4138   n.s.    0.000
  Overselling        9.238       0.0099     **    0.268
  Irrelevance        1.551       0.4605   n.s.    0.000

  Dunn post-hoc within High SIRS:

    Total
    no_context vs context_only              0.0020     **
    no_context vs kg_context                0.0001    ***
    context_only vs kg_context              1.0000   n.s.

    Vagueness
    no_context vs context_only              0.0023     **
    no_context vs kg_context                0.0023     **
    context_only v

In [28]:
# ── Dunn post-hoc within each richness group ──────────────────────────────────
print("\n" + "=" * 70)
print("DUNN POST-HOC — Pairwise comparisons within each richness group")
print("(Bonferroni corrected)")
print("=" * 70)

pairs = [("no_context", "context_only"),
         ("no_context", "kg_context"),
         ("context_only", "kg_context")]

for richness_level in RICHNESS:
    df_sub = df_rich[df_rich["Documentation_richness"] == richness_level].copy()
    df_sub = df_sub[df_sub["grounding_condition"].isin(CONDITIONS)].copy()

    print(f"\n{'=' * 70}")
    print(f"  {richness_level} SIRS companies (n = {len(df_sub)})")
    print(f"{'=' * 70}")
    print(f"  {'Dimension':<15} {'NC vs URAG':>12} {'NC vs KG':>12} {'URAG vs KG':>12}")
    print(f"  {'-' * 55}")

    for col, label in zip(all_cols, all_labels):
        try:
            dunn = posthoc_dunn(df_sub, val_col=col,
                                group_col="grounding_condition",
                                p_adjust="bonferroni")
            p_nc_urag  = dunn.loc["no_context",   "context_only"]
            p_nc_kg    = dunn.loc["no_context",   "kg_context"]
            p_urag_kg  = dunn.loc["context_only", "kg_context"]

            def fmt(p):
                if p < 0.001:
                    return f"{p:.4f}***"
                elif p < 0.01:
                    return f"{p:.4f}**"
                elif p < 0.05:
                    return f"{p:.4f}*"
                else:
                    return f"{p:.4f} n.s."

            print(f"  {label:<15} {fmt(p_nc_urag):>12} {fmt(p_nc_kg):>12} {fmt(p_urag_kg):>12}")

        except Exception as e:
            print(f"  {label:<15} {'ERROR':>12} {str(e)}")


DUNN POST-HOC — Pairwise comparisons within each richness group
(Bonferroni corrected)

  High SIRS companies (n = 30)
  Dimension         NC vs URAG     NC vs KG   URAG vs KG
  -------------------------------------------------------
  Total               0.0020**    0.0001***  1.0000 n.s.
  Vagueness           0.0023**     0.0023**  1.0000 n.s.
  Misleading          0.0012**    0.0000***  0.9646 n.s.
  Concealment      1.0000 n.s.  0.5595 n.s.  1.0000 n.s.
  Overselling      0.2621 n.s.     0.0073**  0.5589 n.s.
  Irrelevance      1.0000 n.s.  0.6461 n.s.  1.0000 n.s.

  Low SIRS companies (n = 30)
  Dimension         NC vs URAG     NC vs KG   URAG vs KG
  -------------------------------------------------------
  Total            0.6112 n.s.     0.0018**  0.0931 n.s.
  Vagueness        1.0000 n.s.  1.0000 n.s.  1.0000 n.s.
  Misleading       1.0000 n.s.    0.0001***     0.0033**
  Concealment      0.7855 n.s.  1.0000 n.s.  0.4641 n.s.
  Overselling          0.0121*     0.0025**  1.00